# Aggregating Data

In this notebook, we will merge the following data files: 

- VarroaPathogen_quarterly_agg_215on.csv which contains the processed data from APHIS. 
- bee_data_cleaned.csv which contains the processed USDA NASS survey
- fall-2025-bee-health-early-warning-system/data/interim/normalized_weather_adv_data.csv which contains processed weather data from NOAA 



In [ ]:
# Load necessary libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import us

Start by loading all the data sets and renaming columns so that they are consistent across data sets. 

In [ ]:
nass_df = pd.read_csv("../data/interim/bee_data_cleaned.csv", na_values=["NaN"])

In [ ]:
nass_df.sample(5)

In [ ]:
# shorten some column names: 
nass_df = nass_df.rename(columns={
    'Year': 'year', 
    'Period':'quarter', 
    'State': 'state',
    'HONEY, BEE COLONIES - ADDED & REPLACED, MEASURED IN COLONIES': 'num_added_replaced', 
    'HONEY, BEE COLONIES - INVENTORY, MAX, MEASURED IN COLONIES': 'max_inventory', 
    'COLONIES AT FIRST OF QUARTER': 'count_on_1stday', 
    'HONEY, BEE COLONIES - LOSS, DEADOUT, MEASURED IN COLONIES': 'num_loss', 
    'HONEY, BEE COLONIES - LOSS, DEADOUT, MEASURED IN PCT OF COLONIES': 'pct_loss',
    'HONEY, BEE COLONIES, AFFECTED BY DISEASE - INVENTORY, MEASURED IN PCT OF COLONIES': 'pct_disease', 
    'HONEY, BEE COLONIES, AFFECTED BY OTHER CAUSES - INVENTORY, MEASURED IN PCT OF COLONIES': 'pct_other',
    'HONEY, BEE COLONIES, AFFECTED BY PESTICIDES - INVENTORY, MEASURED IN PCT OF COLONIES': 'pct_pesticides',
    'HONEY, BEE COLONIES, AFFECTED BY PESTS ((EXCL VARROA MITES)) - INVENTORY, MEASURED IN PCT OF COLONIES':'pct_pests_nomites',
    'HONEY, BEE COLONIES, AFFECTED BY UNKNOWN CAUSES - INVENTORY, MEASURED IN PCT OF COLONIES':'pct_unknown',
    'HONEY, BEE COLONIES, AFFECTED BY VARROA MITES - INVENTORY, MEASURED IN PCT OF COLONIES':'pct_varroa',
    'HONEY, BEE COLONIES, RENOVATED - INVENTORY, MEASURED IN COLONIES':'num_renovated',
    'HONEY, BEE COLONIES, RENOVATED - INVENTORY, MEASURED IN PCT OF COLONIES':'pct_renovated'
})

In [ ]:
# Map full name to abbreviation
nass_df["state_code"] = nass_df["state"].map(lambda x: us.states.lookup(x).abbr if us.states.lookup(x) else None)

In [ ]:
# Load APHIS data
aphis_df = pd.read_csv('../data/interim/VarroaPathogen_quarterly_agg_2015onwards.csv')
aphis_df.sample(10)

In [ ]:
# rename a few columns for consistency
aphis_df = aphis_df.rename(columns={
    'sample_year':'year', 
    'sample_quarter': 'quarter'}
)

In [ ]:
aphis_df = aphis_df.drop(columns=['sample_month_number']) # hold over from unaggergated data set

Include a column for the region associated with each state. 

In [ ]:
state_to_region = {
    "AL": "Southeast",
    "AK": "Northwest", # does not align climactically with other regions, APHIS has no data for Alaska, but 
    "AZ": "Southwest",
    "AR": "South",
    "CA": "West",
    "CO": "Southwest",
    "CT": "Northeast",
    "DE": "Northeast",
    "FL": "Southeast",
    "GA": "Southeast",
    "HI": "West", # choice made because of: oceanic influence, mild temperatures, winter wet season, and dry summer months.
    "ID": "Northwest",
    "IL": "Ohio Valley",
    "IN": "Ohio Valley",
    "IA": "Upper Midwest",
    "KS": "South",
    "KY": "Ohio Valley",
    "LA": "South",
    "ME": "Northeast",
    "MD": "Northeast",
    "MA": "Northeast",
    "MI": "Upper Midwest",
    "MN": "Upper Midwest",
    "MS": "Southeast",
    "MO": "South",
    "MT": "Northern Rockies & Plains",
    "NE": "Northern Rockies & Plains",
    "NV": "Southwest",
    "NH": "Northeast",
    "NJ": "Northeast",
    "NM": "Southwest",
    "NY": "Northeast",
    "NC": "Southeast",
    "ND": "Northern Rockies & Plains",
    "OH": "Ohio Valley",
    "OK": "South",
    "OR": "Northwest",
    "PA": "Northeast",
    "RI": "Northeast",
    "SC": "Southeast",
    "SD": "Northern Rockies & Plains",
    "TN": "Southeast",
    "TX": "South",
    "UT": "Southwest",
    "VT": "Northeast",
    "VA": "Southeast",
    "WA": "Northwest",
    "WV": "Ohio Valley",
    "WI": "Upper Midwest",
    "WY": "Northern Rockies & Plains"
}


In [ ]:
aphis_df['region']=aphis_df['state_code'].map(state_to_region)
nass_df['region']=nass_df['state_code'].map(state_to_region)

In [ ]:
# Load weather data
weather_df = pd.read_csv("../data/interim/normalized_weather_adv_data.csv", na_values=["NaN"])

In [ ]:
weather_df.sample(5)

In [ ]:
# rename some columns for consistency
weather_df = weather_df.rename(columns={
    'Year': 'year', 
    'Quarter': 'quarter'}
)

In [ ]:
# Map full name to abbreviation
weather_df["state_code"] = weather_df["state_name"].map(lambda x: us.states.lookup(x).abbr if us.states.lookup(x) else None)


## Combining data frames 

Now we are ready to combine data frames. The APHIS data contains approximately 1000 rows, which is roughly 800 less than the NASS set. We will therefore create two data sets: 

1. Only include rows for which there is data from both sets. We will call this bee_combined_only_complete.csv
2. Include all rows from the NASS data set and leave missing data as NaN. We will call this bee_combined_all.csv



In [ ]:
nass_aphis_both_df = pd.merge(
    nass_df,
    aphis_df,
    on=['year', 'quarter', 'state_code', 'region'],
    how='inner'
)

bee_complete_df = pd.merge(
    nass_aphis_both_df,
    weather_df,
    on=['year', 'quarter', 'state_code'],
    how='left'
)


In [ ]:
bee_complete_df.columns

In [ ]:
nass_aphis_all_df = pd.merge(
    nass_df,
    aphis_df,
    on=['year', 'quarter', 'state_code', 'region'],
    how='left'
)

bee_all_df = pd.merge(
    nass_aphis_all_df,
    weather_df,
    on=['year', 'quarter', 'state_code'],
    how='left'
)


In [ ]:
bee_all_df.info()

# Target: Risk of Colony Loss

We want to be able to use the data from one quarter to predict whether the next quarter will have a higher than average rate of colony collapse. We define the average risk to be the median risk over the past 5 years for a given state and quarter. We will use the NASS data where available. For year 2015-2019 for historic data, we will use an estimate of quarterly loss from the BIP seasonal loss data to fill in historic years. 


We will start by loading the BIP data (for historic loss). 

In [31]:
bip_df=pd.read_csv("../data/interim/quarterly_bip_df.csv")

In [32]:
bip_df['pct_loss'] = bip_df['loss_quarter']*100

In [33]:
bip_df.sample(10)

,state_code,year,quarter,loss_quarter,region,pct_loss
2262,FL,2019,Q2,0.155863,Southeast,15.586325
1382,NH,2014,Q4,0.343389,Northeast,34.338921
231,IN,2009,Q3,NaN,Ohio Valley,NaN
786,OR,2011,Q4,0.140208,Northwest,14.020777
796,TN,2011,Q4,0.097027,Southeast,9.702742
2548,MSO,2020,Q2,0.212065,NaN,21.206497
1360,LA,2014,Q4,0.132182,South,13.218228
268,NC,2009,Q2,NaN,Southeast,NaN
1778,MN,2016,Q4,0.231430,Upper Midwest,23.143014
3116,NH,2023,Q2,0.116765,Northeast,11.676463


In [34]:
# Start by filling all the NaN values in pct_loss with the regional mean for that year and quarter in both the BIP data and the NASS data. 
bee_all_df['pct_loss'] = bee_all_df['pct_loss'].fillna(
    bee_all_df.groupby(['region', 'year', 'quarter'])['pct_loss'].transform('mean')
)

bee_complete_df['pct_loss'] = bee_complete_df['pct_loss'].fillna(
    bee_complete_df.groupby(['region', 'year', 'quarter'])['pct_loss'].transform('mean')
)

In [35]:
def add_rolling_median_loss(df, T=5):
    """
    Adds a rolling median column to df using bee_all_df and bip_df as sources.
    
    df: pandas DataFrame with at least ['state_code','year','quarter','pct_loss']
    T: number of years to look back for the rolling median
    """
    def get_rolling_median_for_row(row):
        state = row['state_code']
        year = row['year']
        quarter = row['quarter']

        years_needed = list(range(year-T, year))

        bee_years = [y for y in years_needed if y >= 2020]
        bip_years = [y for y in years_needed if y < 2020]

        bee_vals = bee_all_df.loc[
            (bee_all_df['state_code'] == state) &
            (bee_all_df['quarter'] == quarter) &
            (bee_all_df['year'].isin(bee_years)),
            'pct_loss'
        ]

        bip_vals = bip_df.loc[
            (bip_df['state_code'] == state) &
            (bip_df['quarter'] == quarter) &
            (bip_df['year'].isin(bip_years)),
            'pct_loss'
        ]

        all_vals = pd.concat([bee_vals, bip_vals])
        return all_vals.median() if not all_vals.empty else pd.NA

    df['5yr_median_loss'] = df.apply(lambda row: get_rolling_median_for_row(row), axis=1)
    return df



In [36]:
bee_all_df = add_rolling_median_loss(bee_all_df)
bee_complete_df = add_rolling_median_loss(bee_complete_df)


In [37]:
bee_all_df.sample(5)

,year,quarter,state,num_added_replaced,max_inventory,count_on_1stday,num_loss,pct_loss,pct_disease,pct_other,...,Num_Frost_Days,M1_avg_tvol,M2_avg_tvol,M3_avg_tvol,M1_max_tvol,M2_max_tvol,M3_max_tvol,Q_tvol_avg,Num_high_vol_days,5yr_median_loss
432,2016,Q3,TEXAS,"12,500","143,000",129000,"12,000",8.0,1.70,3.70,...,0.010753,0.550348,0.590279,0.627230,0.707426,0.535077,0.512063,0.439343,0.000000,16.174488
1831,2022,Q2,TEXAS,"42,000","415,000",345000,"46,000",11.0,0.05,5.10,...,0.021505,0.676693,0.732854,0.729289,0.680538,0.621284,0.686369,0.624546,0.114286,9.0
2223,2023,Q4,NEW JERSEY,530,"23,000",23000,770,3.0,0.05,0.05,...,0.354839,0.439089,0.601377,0.467707,0.165813,0.370392,0.234017,0.313056,0.000000,5.0
1323,2020,Q2,TENNESSEE,"4,200","13,500",13500,"1,200",9.0,0.05,3.30,...,0.032258,0.552721,0.594720,0.625375,0.427657,0.466706,0.493969,0.440998,0.028571,6.789168
1272,2020,Q1,VERMONT,500,"7,000",6500,520,7.0,0.60,2.00,...,0.913978,0.323043,0.537983,0.582096,0.149808,0.246730,0.276236,0.290339,0.057143,17.809504


Now that the median loss is computed. We will finally add the target column: 'pct_loss_above_next_median' which will check if the percent loss for the following quarter loss is higher than the 5 year median for that quarter. 

In [39]:
def add_next_quarter_flag(df, pct_col='pct_loss', median_col='5yr_median_loss'):
    """
    Adds a binary column indicating if the next quarter's pct_loss is above
    the next quarter's rolling median.
    
    df: DataFrame with at least ['state_code', 'year', 'quarter', pct_col, median_col]
    pct_col: name of the column with actual values
    median_col: name of the column with 5-year median values
    
    Returns: DataFrame with a new column 'pct_loss_above_next_median'
    """
    
    # Make a copy to avoid modifying original
    df = df.copy()
    
    # Ensure quarters are sorted correctly
    quarter_order = ['Q1','Q2','Q3','Q4']
    df['quarter_num'] = df['quarter'].map({q: i for i, q in enumerate(quarter_order, start=1)})
    
    def compute_next_quarter_flag(group):
        group = group.sort_values(['year', 'quarter_num']).copy()
        group['pct_next'] = group[pct_col].shift(-1)
        group['median_next'] = group[median_col].shift(-1)
        group['pct_loss_above_next_median'] = (group['pct_next'] > group['median_next']).astype(int)
        return group

    df = df.groupby('state_code', group_keys=False).apply(compute_next_quarter_flag)
    
    # Clean up helper columns
    df = df.drop(columns=['quarter_num', 'pct_next', 'median_next'])
    
    return df
